In [4]:
require(data.table)
require(tidyverse)
require(phyloseq)
require(ggplot2)
require(RColorBrewer)
require(metacoder)
require(vegan)
require(DESeq2)

# using vegan to map out stats and ordination plots for 16S sequences

In [5]:
ps<-readRDS(file = "/work/pi_sarah_gignouxwolfsohn_uml_edu/caroline/BEL_16S_ITS2/BEL_16S_outputs/ps_16S.rds")
#removing any taxa that don't show up in any samples to speed up the process
ps <- prune_taxa(taxa_sums(ps) > 0, ps)
ps

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 193447 taxa and 470 samples ]
sample_data() Sample Data:       [ 470 samples by 9 sample variables ]
tax_table()   Taxonomy Table:    [ 193447 taxa by 6 taxonomic ranks ]

In [6]:
#normalizing ps by converting rawcounts into relative abundances
#so samples with more reads wont be over represented
#using ps bc only to the count data (OTU table), while preserving the rest of the object
ps_norm = transform_sample_counts(ps, function(x) 1E6 * x / sum(x))

In [7]:
#isolate just bacteria
ps_norm_bac=subset_taxa(ps_norm, Kingdom=="Bacteria")
#remove chloroplast order
ps_norm_nochlo=subset_taxa(ps_norm_bac, Order!="Chloroplast")
#remove mitochondria family
ps_norm_nomit=subset_taxa(ps_norm_nochlo, Family!="Mitochondria")
ps_norm_nomit

phyloseq-class experiment-level object
otu_table()   OTU Table:         [ 64055 taxa and 470 samples ]
sample_data() Sample Data:       [ 470 samples by 9 sample variables ]
tax_table()   Taxonomy Table:    [ 64055 taxa by 6 taxonomic ranks ]

In [8]:
# convert the sample_data() within a phyloseq object to a vegan compatible data object
pssd2veg <- function(ps_norm_nomit) {
  sd_nomit <- sample_data(ps_norm_nomit)
  return(as(sd_nomit,"data.frame"))
}
#using phyloseq nmds plot no chloroplast
sample_nomit <- pssd2veg(ps_norm_nomit)

In [10]:
# convert the otu_table() within a phyloseq object to a vegan compatible data object
psotu2veg <- function(ps_norm_nomit) {
  otu_nomit <- otu_table(ps_norm_nomit)
  if (taxa_are_rows(otu_nomit)) {
    otu_nomit <- t(otu_nomit)
  }
  return(as(otu_nomit, "matrix"))
}

# Extract normalized OTU matrix and sample data
otu_nomit <- psotu2veg(ps_norm_nomit)

### clean sample metadata

In [11]:
sample_nomit <- as.data.frame(sample_data(ps_norm_nomit))
#save sammple names as a column so tidy doesn't get rid of it during filtering
sample_nomit$SampleID <- rownames(sample_nomit)

In [13]:
head(sample_nomit)

,Health_status,colony,Date_16S,Condition,Species,Month_year,Seq_run,Transect,DateFormatted,SampleID
,<chr>,<chr>,<chr>,<chr>,<chr>,<fct>,<int>,<chr>,<date>,<chr>
012024_BEL_CBC_T1_557_SSID,Healthy,1_3,3_4_2025,Healthy,SSID,Jan 2024,1,CBC30N,2024-01-10,012024_BEL_CBC_T1_557_SSID
012024_BEL_CBC_T1_559_MCAV,Healthy,1_24,8_22_2025,Healthy,MCAV,Jan 2024,2,CBC30N,2024-01-10,012024_BEL_CBC_T1_559_MCAV
012024_BEL_CBC_T1_563_PSTR,Healthy,1_12,3_6_2025,Healthy,PSTR,Jan 2024,1,CBC30N,2024-01-10,012024_BEL_CBC_T1_563_PSTR
012024_BEL_CBC_T1_565_PAST,Bleached_Tissue,1_21,2_10_2026,CLP,PAST,Jan 2024,4,CBC30N,2024-01-10,012024_BEL_CBC_T1_565_PAST
012024_BEL_CBC_T2_585_OFAV,Healthy,2_76,2_10_2026,Healthy,OFAV,Jan 2024,4,SR30N,2024-01-12,012024_BEL_CBC_T2_585_OFAV
012024_BEL_CBC_T2_587_SSID,Bleached_Tissue,2_72,1_8_2026,CLB,SSID,Jan 2024,3,SR30N,2024-01-12,012024_BEL_CBC_T2_587_SSID


In [14]:
## check to make sure meta and otu table are still compatible

# Should return TRUE
all(rownames(sample_nomit) == rownames(otu_nomit))

[1] TRUE

# vegan cluster analysis

In [15]:
# merge otu and sam_clean by sample ID
otu_nomit$SampleID <- rownames(otu_nomit)
sample_nomit$SampleID <- rownames(sample_nomit)

Warning message in otu_nomit$SampleID <- rownames(otu_nomit):
“Coercing LHS to a list”


In [ ]:
otu_merged <- merge(otu_nomit, sample_nomit, by = "SampleID",
                    all = TRUE, sort = FALSE)
rownames(otu_merged) <- otu_merged$SampleID
otu_merged$SampleID <- NULL
otu_nomit$SampleID <- NULL

In [ ]:
class(otu_merged)
head(otu_merged)
nrow(otu_nomit)
nrow(otu_merged)
all(rownames(otu_nomit) == rownames(otu_merged))

## PC clustering of host species with hellinger distance

In [ ]:
# Hellinger distance, comparable to euclidean 
ord <- decostand(otu_nochlo, method = "hellinger")

In [ ]:
str(otu_merged$Species)
table(otu_merged$Species)

In [ ]:
# plot settings
options(repr.plot.width=20, repr.plot.height=18)

In [ ]:
#by species 
disp <- "sites" 
scl <- "symmetric" 

#transforming species col from chr to factor 
otu_merged$species <- factor(otu_merged$species)
# PCA via rda()
pca_mod <- rda(ord)
#color points
# Color vector
col_vec <-c("red", "blue", "orange", "grey", "purple", "green")
cols <- col_vec[otu_merged$species]
plot(pca_mod, type = "n", scaling = scl, display = disp) 
ordihull(pca_mod, groups = otu_merged$species, col = col_vec, scaling = scl, lwd = 2) 
ordispider(pca_mod, groups = otu_merged$species, col = col_vec, scaling = scl, label = TRUE) 
points(pca_mod, display = disp, scaling = scl, pch = 21, col = "red", bg = "yellow")

## dendogram cluster analysis

In [ ]:
dij <- vegdist(otu_nochlo) ## bray curtis dissimilarity
clu <- hclust(dij, method = "average")
# 2 clusters bc I know Date_16S is already driving into 2 clusters
grp <- cutree(clu, 8)

In [ ]:
# visualizing the parent dendogram
plot(clu); rect.hclust(clu, k=8, border="red")

## heatmap with dendogram